# Training Data-to-Data Flow Matching – FashionMNIST

## Overview

This notebook trains a **Rectified Flow model** using the **Data-to-Data Flow Matching objective** on FashionMNIST.

**Key Setup:**
- Source class: **Sneaker** (class 7)
- Target class: **Ankle Boot** (class 9)
- Model: **DiT_Llama** (medium profile)
- No conditioning signal – the model learns pure transport between distributions

**Checkpoint Handling:**
- Saves both `latest` and `best` checkpoints locally to `assets/fmnist_d2d_runs/`
- Works seamlessly in local VS Code and Google Colab
- Can resume training from best checkpoint

## Training Objective

The model learns to approximate the velocity field $v_\theta(z_t, t)$ that transports samples from the source to target distribution:

$$\mathcal{L} = \mathbb{E}_{x_A, x_B, t}\left[ \| v_\theta(z_t, t) - (x_B - x_A) \|^2 \right]$$

where $z_t = (1-t)x_A + t x_B$ is the linear interpolation path.

In [1]:
import os
import sys
import random
import subprocess
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import numpy as np

# Fix random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Detect device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# === Setup repo path (like Notebook 5) ===
# Detect Colab environment
IN_COLAB = "google.colab" in sys.modules
print(f"Running in Colab: {IN_COLAB}")

def find_repo_root():
    """Locate the minRF_interpolation repository root.
    
    Checks multiple candidate paths to support both local VS Code
    and Google Colab environments.
    """
    candidates = []

    # Respect environment variable override
    env_repo = os.environ.get("MINRF_REPO")
    if env_repo:
        candidates.append(Path(env_repo).expanduser())

    # Check well-known locations
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd / "minRF_interpolation",
        cwd.parent / "minRF_interpolation",
        Path("/content/minRF_interpolation"),
        Path("/content/drive/MyDrive/minRF_interpolation"),
        Path("/content/Projects/minRF_interpolation"),
        Path.home() / "Projects" / "minRF_interpolation",
        Path("/home/felicitasl/Projects/minRF_interpolation"),
    ])

    # Walk up directory tree as fallback
    for parent in [cwd, *cwd.parents]:
        candidates.append(parent / "minRF_interpolation")

    # Test each candidate for presence of dit.py
    seen = set()
    for path in candidates:
        p = path.expanduser().resolve()
        if str(p) in seen:
            continue
        seen.add(str(p))
        if (p / "dit.py").exists():
            return p

    return None


repo_root = find_repo_root()

# In Colab, clone the repository if not found locally
if repo_root is None and IN_COLAB:
    repo_url = os.environ.get("MINRF_REPO_URL", "https://github.com/felicitasl/minRF_interpolation.git")
    clone_target = Path("/content/minRF_interpolation")

    if not clone_target.exists():
        print(f"Repository not found. Attempting to clone from {repo_url}...")
        try:
            subprocess.run(["git", "clone", repo_url, str(clone_target)], check=True, timeout=120)
            repo_root = find_repo_root()
        except Exception as e:
            print(f"   Clone failed: {e}")
    else:
        repo_root = clone_target if (clone_target / "dit.py").exists() else None

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate minRF_interpolation repository. "
        "Set MINRF_REPO environment variable or run from within the repo. "
        f"Checked cwd={Path.cwd().resolve()}"
    )

print(f"  Using repo_root: {repo_root}")

# Add to sys.path
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Export as REPO_ROOT for downstream use
REPO_ROOT = repo_root

In [2]:
# === Import API modules ===
# Note: Only import after repo_path is set & added to sys.path

try:
    from api.training_api import TrainingSpec, train_data_to_data_flow
    print("✓ Successfully imported training API")
except ImportError as e:
    print(f"Import failed: {e}")
    print(f"sys.path: {sys.path[:3]}")  # Show first few paths for debugging
    raise

✓ Successfully imported training API


## Training Configuration

Adjust these parameters as needed -> Default settings train on a single GPU/CPU.

In [3]:
# === Training Specification ===

train_spec = TrainingSpec(
    model_profile="medium",           # Model size: small, medium, or large
    source_class=7,                    # Sneaker
    target_class=9,                    # Ankle Boot
    root=str(REPO_ROOT / "data"),     # Dataset root (absolute path)
    batch_size=128,                    # Training batch size
    test_batch_size=128,               # Validation batch size
    checkpoint_dir=str(REPO_ROOT / "assets" / "fmnist_d2d_runs"),  # Absolute path to save checkpoints
    train_epochs=100,                  # Max epochs
    quick_run=False,                   # Set to True for debugging (8 epochs, 250 batches)
    force_retrain=False,               # Force restart from scratch
    load_best_if_available=True,       # Resume from best checkpoint if it exists
    early_stopping=True,               # Stop if validation loss plateaus
    patience=10,                       # Epochs to wait for improvement
    min_delta=1e-4,                    # Minimum improvement threshold
    lr=2e-4,                           # Learning rate
    model_name="fmnist_d2d",          # Checkpoint name prefix
)

print("Training Configuration:")
print(f"  Source -> Target: {train_spec.source_class} -> {train_spec.target_class}")
print(f"  Model Profile: {train_spec.model_profile}")
print(f"  Batch Size: {train_spec.batch_size}")
print(f"  Max Epochs: {train_spec.train_epochs}")
print(f"  Early Stopping: {train_spec.early_stopping} (patience={train_spec.patience})")
print(f"  Checkpoint Dir: {train_spec.checkpoint_dir}")
print(f"  Checkpoints will be saved to repo: {REPO_ROOT}")

Training Configuration:
  Source → Target: 7 → 9
  Model Profile: medium
  Batch Size: 128
  Max Epochs: 100
  Early Stopping: True (patience=10)
  Checkpoint Dir: /content/minRF_interpolation/assets/fmnist_d2d_runs
✓ Checkpoints will be saved to repo: /content/minRF_interpolation


## Training

In [4]:
print("="*80)
print("STARTING TRAINING: Data-to-Data Flow Matching")
print("="*80)
print()

# Run training
result = train_data_to_data_flow(spec=train_spec, device=device)

print()
print("="*80)
print("TRAINING COMPLETED")
print("="*80)

STARTING TRAINING: Data-to-Data Flow Matching



100%|██████████| 26.4M/26.4M [00:01<00:00, 19.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 304kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.65MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 19.2MB/s]


new best checkpoint at epoch 1: val=0.19298
new best checkpoint at epoch 2: val=0.13822
new best checkpoint at epoch 3: val=0.11553
new best checkpoint at epoch 4: val=0.09678
new best checkpoint at epoch 5: val=0.08313
new best checkpoint at epoch 6: val=0.07733
new best checkpoint at epoch 7: val=0.06998
new best checkpoint at epoch 9: val=0.06665
new best checkpoint at epoch 10: val=0.06586
new best checkpoint at epoch 11: val=0.06102
new best checkpoint at epoch 13: val=0.05841
new best checkpoint at epoch 14: val=0.05429
new best checkpoint at epoch 19: val=0.05263
new best checkpoint at epoch 20: val=0.05076
new best checkpoint at epoch 26: val=0.04754
new best checkpoint at epoch 31: val=0.04555
new best checkpoint at epoch 32: val=0.04526
new best checkpoint at epoch 36: val=0.04494
new best checkpoint at epoch 38: val=0.04417
new best checkpoint at epoch 43: val=0.04366
new best checkpoint at epoch 47: val=0.04215
new best checkpoint at epoch 50: val=0.04149
new best checkpoin

In [5]:
# Extract results
train_history = result["train_history"]
val_history = result["val_history"]
best_val_loss = result["best_val_loss"]
latest_path = result["latest_path"]
best_path = result["best_path"]

print(f"\nTraining Summary:")
print(f"  Total Epochs Trained: {len(train_history)}")
print(f"  Final Training Loss: {train_history[-1]:.6f}")
print(f"  Final Validation Loss: {val_history[-1]:.6f}")
print(f"  Best Validation Loss: {best_val_loss:.6f}")
print(f"\nCheckpoints Saved:")
print(f"  Latest: {latest_path}")
print(f"  Best:   {best_path}")
print(f"  Exists (best): {best_path.exists()}")
print(f"  Exists (latest): {latest_path.exists()}")

## Training Loss Visualization

In [6]:
# Plot training and validation curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Full training history
epochs = list(range(1, len(train_history) + 1))
ax1.plot(epochs, train_history, label="Training Loss", marker="o", markersize=3, linewidth=2)
ax1.plot(epochs, val_history, label="Validation Loss", marker="s", markersize=3, linewidth=2)
ax1.axhline(y=best_val_loss, color="g", linestyle="--", alpha=0.5, label=f"Best Val = {best_val_loss:.6f}")
ax1.set_xlabel("Epoch", fontsize=12)
ax1.set_ylabel("Loss", fontsize=12)
ax1.set_title("Training & Validation Loss Over Time", fontsize=13, fontweight="bold")
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: Log scale (to see early behavior better)
ax2.semilogy(epochs, train_history, label="Training Loss", marker="o", markersize=3, linewidth=2)
ax2.semilogy(epochs, val_history, label="Validation Loss", marker="s", markersize=3, linewidth=2)
ax2.set_xlabel("Epoch", fontsize=12)
ax2.set_ylabel("Loss (log scale)", fontsize=12)
ax2.set_title("Loss Curves (Log Scale)", fontsize=13, fontweight="bold")
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, which="both")

plt.tight_layout()
plt.savefig(str(REPO_ROOT / "assets" / "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"Training curves saved to assets/training_curves.png")

## Checkpoint Verification

In [8]:
import torch

print("Verifying checkpoint integrity...\n")

# Load best checkpoint and inspect contents
if best_path.exists():
    checkpoint = torch.load(best_path, map_location=device)
    
    print(f"  Best checkpoint loaded: {best_path}")
    print(f"  Checkpoint keys: {list(checkpoint.keys())}")
    
    if "config" in checkpoint:
        config = checkpoint["config"]
        print(f"\n  Model Configuration:")
        for key, value in config.items():
            print(f"    {key}: {value}")
    
    if "model_state_dict" in checkpoint:
        state = checkpoint["model_state_dict"]
        total_params = sum(p.numel() for p in state.values() if isinstance(p, torch.Tensor))
        print(f"\n  Model Parameters: {total_params:,}")
    
    if "train_history" in checkpoint and "val_history" in checkpoint:
        print(f"\n  History:")
        print(f"    Epochs recorded: {len(checkpoint['train_history'])}")
        print(f"    Final train loss: {checkpoint['train_history'][-1]:.6f}")
        print(f"    Final val loss: {checkpoint['val_history'][-1]:.6f}")
else:
    print(f"Best checkpoint not found at {best_path}")

In [ ]:
from google.colab import files

# download best checkpoint
files.download(str(best_path))

# download newest checkpoint
files.download(str(latest_path))

# otherwise if colab extension in vsc: download both via the left colab panel